# 7주차 OCR 실습 — 포켓몬 카드에서 이름·HP 읽기
**세대가 다른 OCR 3종 비교** (강의 19쪽 발전 과정을 실험 하나로)

| 세대 | 모델 | 크기 | 방식 |
|---|---|---|---|
| 딥러닝 OCR (CRAFT + CRNN) | **EasyOCR** | 수 MB | 글자 검출 → 인식 후, 규칙으로 이름·HP 추출 |
| VLM 문서 AI (문서 특화) | **GLM-OCR** (`zai-org/GLM-OCR`) | 0.9B | JSON 스키마 정보 추출 프롬프트 |
| 범용 VLM | **Qwen3.5-9B** (`Qwen/Qwen3.5-9B`) | 9B | 자연어 지시 → JSON |

데이터: Hugging Face `TheFusion21/PokemonCards` (카드 이미지 URL + 정답 이름·HP) 중 무작위 200장.
**실행**: `런타임 > 런타임 유형 변경 > L4 GPU` → `런타임 > 모두 실행` (약 20~30분). 마지막 셀이 `results_w7.zip`을 받고 결과 JSON을 출력합니다.

## 1. 환경 설정

In [ ]:
!pip -q install -U transformers accelerate easyocr datasets
import os, re, json, time, random, zipfile, gc, unicodedata
import numpy as np, torch, requests, transformers, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor
from datasets import load_dataset

device = "cuda"
ENV = {"gpu": torch.cuda.get_device_name(0), "vram_GB": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),
       "torch": torch.__version__, "transformers": transformers.__version__}
print(ENV)
SEED, N_CARDS, MAX_SIDE = 42, 200, 1024
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
os.makedirs("results_w7", exist_ok=True); os.makedirs("cards", exist_ok=True)

## 2. 데이터: 포켓몬 카드 200장 샘플링

In [ ]:
ds = load_dataset("TheFusion21/PokemonCards", split="train")
df = ds.to_pandas()
df = df[df["hp"].astype(str).str.fullmatch(r"\d+")].reset_index(drop=True)      # HP가 숫자인 카드만
sample = df.sample(n=N_CARDS * 2, random_state=SEED).reset_index(drop=True)      # 다운로드 실패 대비 여유분

def fetch(row):
    path = f"cards/{row['id']}.png"
    try:
        if not os.path.exists(path):
            r = requests.get(row["image_url"], timeout=30); r.raise_for_status()
            img = Image.open(BytesIO(r.content)).convert("RGB")
            img.thumbnail((MAX_SIDE, MAX_SIDE)); img.save(path)
        return path
    except Exception as e:
        return None
with ThreadPoolExecutor(16) as ex:
    paths = list(ex.map(fetch, [r for _, r in sample.iterrows()]))
sample["path"] = paths
cards = sample[sample["path"].notna()].head(N_CARDS).reset_index(drop=True)

SUFFIX = re.compile(r"(\b(EX|ex|GX|V|VMAX|VSTAR|BREAK|LV\.X|Prime|LEGEND|G|GL|FB|C|E4)\b|δ|☆|◇)")
cards["special"] = cards["name"].apply(lambda s: bool(SUFFIX.search(s)))   # EX·GX·V 등 특수 표기 카드
DATA = {"n_total_with_hp": len(df), "n_cards": len(cards), "n_sets": int(cards["set_name"].nunique()),
        "n_special_name": int(cards["special"].sum()),
        "img_size_example": list(Image.open(cards.loc[0, "path"]).size)}
print(DATA); cards[["id", "name", "hp", "set_name", "special"]].head()

## 3. 평가 함수 (이름 정확도 · 글자 오류율 CER · HP 정확도)

In [ ]:
def norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()   # é → e, δ 등 제거
    return re.sub(r"[^a-z0-9]", "", s.lower())

def lev(a, b):
    if a == b: return 0
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j-1] + 1, prev[j-1] + (ca != cb)))
        prev = cur
    return prev[-1]

def cer(pred, gt):
    p, g = norm(pred), norm(gt)
    return min(1.0, lev(p, g) / max(1, len(g)))

def parse_json(text):
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S)
    m = re.search(r"\{.*\}", text, flags=re.S)
    name, hp = "", ""
    if m:
        try:
            j = json.loads(m.group(0)); name, hp = str(j.get("name", "")), str(j.get("hp", ""))
        except Exception:
            pass
    if not name:
        mm = re.search(r'"?name"?\s*[:：]\s*"?([^",}\n]+)', text); name = mm.group(1).strip() if mm else ""
    if not hp:
        mm = re.search(r'"?hp"?\s*[:：]\s*"?(\d+)', text, flags=re.I); hp = mm.group(1) if mm else ""
    hp = re.sub(r"\D", "", hp)
    return name.strip(), hp

def evaluate(preds):
    rows = []
    for (_, c), p in zip(cards.iterrows(), preds):
        rows.append({"id": c["id"], "gt_name": c["name"], "gt_hp": str(c["hp"]), "special": c["special"],
                     "pred_name": p["name"], "pred_hp": p["hp"], "raw": p.get("raw", "")[:300], "sec": p["sec"],
                     "name_ok": norm(p["name"]) == norm(c["name"]), "name_cer": cer(p["name"], c["name"]),
                     "hp_ok": p["hp"] == str(c["hp"])})
    r = pd.DataFrame(rows)
    r["both_ok"] = r["name_ok"] & r["hp_ok"]
    met = {"name_acc": round(r["name_ok"].mean(), 4), "name_CER": round(r["name_cer"].mean(), 4),
           "hp_acc": round(r["hp_ok"].mean(), 4), "both_acc": round(r["both_ok"].mean(), 4),
           "name_acc_plain": round(r.loc[~r["special"], "name_ok"].mean(), 4),
           "name_acc_special": round(r.loc[r["special"], "name_ok"].mean(), 4) if r["special"].any() else None,
           "sec_per_card": round(r["sec"].iloc[3:].mean(), 3)}
    return met, r

RES, TABLES = {}, {}
def free():
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

## 4-1. EasyOCR (CRAFT 검출 + CRNN 인식) + 규칙 기반 추출

In [ ]:
import easyocr
free()
reader = easyocr.Reader(["en"], gpu=True)
STOP = {"basic", "stage", "stage1", "stage2", "hp", "pokemon", "evolves", "from", "put", "on", "the", "trainer"}

def easy_extract(path):
    img = Image.open(path); H = img.height
    det = reader.readtext(np.array(img))                      # [(bbox, text, conf), ...]
    top = [(b, t, c) for b, t, c in det if min(p[1] for p in b) < 0.16 * H]   # 카드 상단 16% 영역 (이름·HP 위치)
    # HP: 'HP' 근처 숫자 또는 상단의 10~340 사이 숫자
    hp = ""
    joined = " ".join(t for _, t, _ in top)
    m = re.search(r"(\d{2,3})\s*HP|HP\s*(\d{2,3})", joined, flags=re.I)
    if m: hp = m.group(1) or m.group(2)
    else:
        nums = [re.sub(r"\D", "", t) for _, t, _ in top if re.fullmatch(r"\D{0,3}\d{2,3}\D{0,3}", t)]
        nums = [n for n in nums if 10 <= int(n) <= 340]
        hp = nums[-1] if nums else ""
    # 이름: 상단에서 글자 높이가 가장 큰 '단어' 텍스트 (숫자·STAGE·BASIC 등 제외)
    cands = []
    for b, t, c in top:
        tt = t.strip()
        if not re.search(r"[A-Za-z]{2,}", tt) or norm(tt) in STOP or re.search(r"\d{2,}", tt): continue
        h = max(p[1] for p in b) - min(p[1] for p in b)
        cands.append((h, tt))
    name = max(cands)[1] if cands else ""
    return {"name": name, "hp": hp, "raw": " | ".join(t for _, t, _ in det)}

preds = []
for i, p in enumerate(cards["path"]):
    t0 = time.perf_counter(); out = easy_extract(p); out["sec"] = time.perf_counter() - t0; preds.append(out)
met, TABLES["EasyOCR"] = evaluate(preds)
RES["EasyOCR"] = {"params": "CRAFT+CRNN (~0.1B 미만)", "peak_vram_GB": round(torch.cuda.max_memory_allocated() / 1e9, 2), **met}
print(RES["EasyOCR"]); del reader; free()

## 4-2. GLM-OCR (0.9B, 문서 특화 VLM) — JSON 스키마 정보 추출

In [ ]:
from transformers import AutoProcessor, AutoModelForMultimodalLM
GLM = "zai-org/GLM-OCR"
try:
    glm_proc = AutoProcessor.from_pretrained(GLM)
except Exception as e:                      # 모델 카드 예시처럼 tokenizer로 대체
    from transformers import AutoTokenizer; print("AutoProcessor 실패 → AutoTokenizer 사용:", e); glm_proc = AutoTokenizer.from_pretrained(GLM)
glm = AutoModelForMultimodalLM.from_pretrained(GLM, dtype=torch.bfloat16, device_map="cuda").eval()
glm_params = sum(p.numel() for p in glm.parameters()) / 1e9
# 모델 카드의 정보 추출 형식: 'JSON 형식으로 출력' 지시 + 스키마
GLM_PROMPT = '请按下列JSON格式输出图中信息:\n{\n    "name": "",\n    "hp": ""\n}'

@torch.no_grad()
def glm_extract(path):
    messages = [{"role": "user", "content": [{"type": "image", "url": os.path.abspath(path)},
                                             {"type": "text", "text": GLM_PROMPT}]}]
    inputs = glm_proc.apply_chat_template(messages, add_generation_prompt=True, tokenize=True,
                                          return_dict=True, return_tensors="pt").to(glm.device)
    out = glm.generate(**inputs, max_new_tokens=96, do_sample=False)
    text = glm_proc.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    name, hp = parse_json(text)
    return {"name": name, "hp": hp, "raw": text}

print("test:", glm_extract(cards.loc[0, "path"]), "| GT:", cards.loc[0, "name"], cards.loc[0, "hp"])
preds = []
for p in cards["path"]:
    torch.cuda.synchronize(); t0 = time.perf_counter(); out = glm_extract(p); torch.cuda.synchronize()
    out["sec"] = time.perf_counter() - t0; preds.append(out)
met, TABLES["GLM-OCR"] = evaluate(preds)
RES["GLM-OCR"] = {"params": f"{glm_params:.2f}B", "peak_vram_GB": round(torch.cuda.max_memory_allocated() / 1e9, 2), **met}
print(RES["GLM-OCR"]); del glm, glm_proc; free()

## 4-3. Qwen3.5-9B (범용 VLM) — 자연어 지시 (thinking 끔)

In [ ]:
QW = "Qwen/Qwen3.5-9B"
qw_proc = AutoProcessor.from_pretrained(QW)
qw = AutoModelForMultimodalLM.from_pretrained(QW, dtype=torch.bfloat16, device_map="cuda").eval()
qw_params = sum(p.numel() for p in qw.parameters()) / 1e9
QW_PROMPT = ("This is a Pokemon trading card. Read the card's printed Pokemon name (including suffixes such as EX, GX, V, VMAX if printed) "
             "and its HP number at the top. Answer ONLY with JSON: {\"name\": \"...\", \"hp\": \"...\"}")

def qw_inputs(path):
    messages = [{"role": "user", "content": [{"type": "image", "url": os.path.abspath(path)},
                                             {"type": "text", "text": QW_PROMPT}]}]
    try:   # thinking 끄기 (지원 안 되면 아래에서 빈 think 블록을 직접 붙임)
        text = qw_proc.apply_chat_template(messages, add_generation_prompt=True, tokenize=False, enable_thinking=False)
    except TypeError:
        text = qw_proc.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    if text.rstrip().endswith("<think>"):
        text = text.rstrip() + "\n\n</think>\n\n"
    elif "</think>" not in text[-40:]:
        text = text + "<think>\n\n</think>\n\n"
    return qw_proc(text=[text], images=[Image.open(path)], return_tensors="pt").to(qw.device)

@torch.no_grad()
def qw_extract(path):
    inputs = qw_inputs(path)
    out = qw.generate(**inputs, max_new_tokens=96, do_sample=False)
    text = qw_proc.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    name, hp = parse_json(text)
    return {"name": name, "hp": hp, "raw": text}

print("test:", qw_extract(cards.loc[0, "path"]), "| GT:", cards.loc[0, "name"], cards.loc[0, "hp"])
preds = []
for p in cards["path"]:
    torch.cuda.synchronize(); t0 = time.perf_counter(); out = qw_extract(p); torch.cuda.synchronize()
    out["sec"] = time.perf_counter() - t0; preds.append(out)
met, TABLES["Qwen3.5-9B"] = evaluate(preds)
RES["Qwen3.5-9B"] = {"params": f"{qw_params:.2f}B", "peak_vram_GB": round(torch.cuda.max_memory_allocated() / 1e9, 2), **met}
print(RES["Qwen3.5-9B"]); del qw, qw_proc; free()

## 5. 결과 비교 · 시각화 (한글)

In [ ]:
# --- 한글 폰트 설정 (코랩) ---
import os, subprocess, matplotlib.pyplot as plt, matplotlib.font_manager as fm
if not os.path.exists("/usr/share/fonts/truetype/nanum/NanumGothic.ttf"):
    subprocess.run("apt-get -qq install -y fonts-nanum", shell=True)
fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
plt.rcParams["font.family"] = "NanumGothic"; plt.rcParams["axes.unicode_minus"] = False

# ===== 한글 시각화 (노트북 5번 셀과 동일한 코드) =====
# 필요한 변수: RES(모델별 결과 dict), TABLES(모델별 예측 DataFrame), cards(카드 DataFrame, 예시 그림용)
import numpy as np, matplotlib.pyplot as plt
from PIL import Image

LABEL = {"EasyOCR": "EasyOCR\n(전통 OCR: 검출+인식)", "GLM-OCR": "GLM-OCR\n(문서 특화 VLM, 1.1B)", "Qwen3.5-9B": "Qwen3.5-9B\n(범용 VLM, 9.4B)"}
names = list(RES.keys()); COL = ["#8d99ae", "#2a9d8f", "#e76f51"]; x = np.arange(len(names))
n_total = len(TABLES[names[0]]); n_special = int(TABLES[names[0]]["special"].sum())
os.makedirs("results_w7", exist_ok=True)

# (1) 정확도 비교
fig, ax = plt.subplots(figsize=(10, 4.8)); w = .2
for k, (key, lab) in enumerate([("name_acc", "이름 정확도"), ("hp_acc", "HP 정확도"),
                                ("both_acc", "이름+HP 모두 정답"), ("name_CER", "이름 글자 오류율 (낮을수록 좋음)")]):
    b = ax.bar(x + (k - 1.5) * w, [RES[n][key] for n in names], w, label=lab)
    ax.bar_label(b, fmt="%.3f", fontsize=7.5)
ax.set_xticks(x); ax.set_xticklabels([LABEL[n] for n in names], fontsize=9)
ax.set_ylim(0, 1.15); ax.set_ylabel("비율 (0~1)"); ax.legend(fontsize=9, ncol=2, loc="upper left")
ax.set_title(f"포켓몬 카드 OCR 정확도 비교 (카드 {n_total}장)")
plt.tight_layout(); plt.savefig("results_w7/fig_accuracy.png", dpi=200); plt.show()

# (2) 속도 vs 정확도
fig, ax = plt.subplots(figsize=(7, 4.5))
for n, c in zip(names, COL):
    ax.scatter(RES[n]["sec_per_card"], RES[n]["both_acc"], s=120 + 40 * RES[n]["peak_vram_GB"], color=c, alpha=.8)
    ax.annotate(f"{n}\n카드당 {RES[n]['sec_per_card']:.2f}초 · GPU {RES[n]['peak_vram_GB']}GB",
                (RES[n]["sec_per_card"], RES[n]["both_acc"]), textcoords="offset points", xytext=(16, -34), fontsize=9)
ax.set_xlabel("카드 1장당 처리 시간 (초, L4 GPU)"); ax.set_ylabel("이름+HP 모두 정답 비율")
ax.set_xlim(0, max(RES[n]["sec_per_card"] for n in names) * 1.5); ax.set_ylim(0, 1.08); ax.grid(alpha=.3)
ax.set_title("속도 vs 정확도 (원 크기 = 최대 GPU 메모리)")
plt.tight_layout(); plt.savefig("results_w7/fig_speed.png", dpi=200); plt.show()

# (3) 이름 유형별 정확도
fig, ax = plt.subplots(figsize=(8, 4.5)); w = .36
b1 = ax.bar(x - w/2, [RES[n]["name_acc_plain"] for n in names], w, label=f"일반 이름 (예: Weedle, {n_total - n_special}장)")
b2 = ax.bar(x + w/2, [RES[n]["name_acc_special"] or 0 for n in names], w, label=f"특수 표기 이름 (EX·GX·V 등, {n_special}장)")
ax.bar_label(b1, fmt="%.2f", fontsize=8); ax.bar_label(b2, fmt="%.2f", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels([LABEL[n] for n in names], fontsize=9); ax.set_ylim(0, 1.15)
ax.set_ylabel("이름 정확도"); ax.legend(fontsize=9, loc="upper left"); ax.set_title("이름 유형별 정확도 — 로고처럼 디자인된 EX·V 표기가 어려운가?")
plt.tight_layout(); plt.savefig("results_w7/fig_special.png", dpi=200); plt.show()

# (4) 카드 예시 + 모델별 예측 (카드 이미지가 있을 때만)
if "cards" in globals() and os.path.exists(cards.loc[0, "path"]):
    show = list(np.random.RandomState(3).choice(len(cards), 6, replace=False))
    fig, axes = plt.subplots(1, 6, figsize=(21, 8))
    for ax, i in zip(axes, show):
        ax.imshow(Image.open(cards.loc[i, "path"])); ax.axis("off")
        lines = [f"정답: {cards.loc[i, 'name']} / HP {cards.loc[i, 'hp']}"]
        for n in names:
            r = TABLES[n].loc[i]; mark = "정답" if r["both_ok"] else "오답"
            lines.append(f"[{mark}] {n}: {str(r['pred_name'])[:20]} / {r['pred_hp'] if str(r['pred_hp']) not in ('', 'nan') else '-'}")
        ax.set_title("\n".join(lines), fontsize=9, loc="left")
    plt.tight_layout(); plt.savefig("results_w7/fig_examples.png", dpi=110); plt.show()

# 오답 사례 저장
for n in names:
    TABLES[n].to_csv(f"results_w7/pred_{n}.csv", index=False)
    print("====", n, "오답 예시"); print(TABLES[n].loc[~TABLES[n]["both_ok"], ["gt_name", "gt_hp", "pred_name", "pred_hp"]].head(8))

## 6. 결과 저장 및 다운로드

In [ ]:
err = {n: TABLES[n].loc[~TABLES[n]["both_ok"], ["id", "gt_name", "gt_hp", "pred_name", "pred_hp"]].head(15).to_dict("records") for n in names}
out = {"env": ENV, "data": DATA, "seed": SEED, "models": RES, "error_examples": err}
json.dump(out, open("results_w7/results.json", "w"), ensure_ascii=False, indent=2)
print(json.dumps(out, ensure_ascii=False, indent=2))
with zipfile.ZipFile("results_w7.zip", "w") as z:
    for f in os.listdir("results_w7"): z.write(f"results_w7/{f}")
try:
    from google.colab import files; files.download("results_w7.zip")
except Exception as e:
    print("왼쪽 파일 탭에서 results_w7.zip을 직접 다운로드하세요:", e)